# user websockets
> Solveit compatible user-defined HTMX websockets.

In [ ]:
#| default_exp user_ws

In [ ]:
#| hide
import json, os
from urllib.parse import urlencode
from fastcore.foundation import store_attr
from fastcore.meta import delegates
from fasthtml.components import Div, Span
from pyskills.core import allow
from dialoghelper import iife_a, js_eval_a, add_html_a, event_get_a
from pote.dutil import wait_for_a
from pote.basic import id_gen
from pote.server import add_mod, serve_app

In [ ]:
#| hide
import asyncio
from fastcore.foundation import patch, AttrDict as AD
from fastcore.xml import to_xml
from fastcore.test import *
from pyskills import doc
from pyskills import *
from fasthtml.core import FastHTML
from fasthtml.components import Samp, Br
from fasthtml.xtend import Script
from dialoghelper.core import add_msg, read_msg, js_run_a, js_eval, add_scr_a
from pote.basic import simple_id
from pote.logger import logger
from pote.dutil import link_msg
from pote.server import stop_server_a, ensure_server, serve_app, setup_loader
from htmx_bridge.core import get_hx, setup_htmx_bridge

In [ ]:
await stop_server_a()
app=rt=p=None

In [ ]:
_new_id = id_gen()

In [ ]:
await setup_htmx_bridge()
hx = get_hx()

## Problem

Solveit injects browser JavaScript that listens globally for HTMX websocket lifecycle events:

```javascript
function updateWSIndicator(type) {
  if (['error', 'close'].includes(type)) {
    htmx.find('#submit_btn').disabled = true;
    htmx.find('#cancel_btn').disabled = true;
  }
  const color = { error: "orange", open: "limegreen", close: "red" }[type] || 'grey';
  const elt = document.getElementById("ws-indicator");
  if (elt) elt.style.backgroundColor = color;
}

htmx.on("htmx:wsOpen", ({ detail: { event} }) => updateWSIndicator(event.type));
htmx.on("htmx:wsClose", ({ detail: { event } }) => updateWSIndicator(event.type));
htmx.on("htmx:wsError", () => updateWSIndicator('error'));
```

That works for Solveit’s own websocket, but it is **too broad**. If a user also uses HTMX’s websocket extension in their own app inside a dialog, their websocket emits the same `htmx:wsOpen`, `htmx:wsClose`, and `htmx:wsError` events.

So when the user’s own websocket server stops, Solveit sees `htmx:wsClose` / `htmx:wsError` and thinks **Solveit’s own backend connection** is broken. The result is:

- Solveit’s WS indicator turns red/orange
- Submit/cancel buttons may be disabled
- Console logs misleading messages like “Lost connection to solveit”
- HTMX may keep retrying the user websocket using Solveit’s global `htmx.config.wsReconnectDelay`

The real problem is not that Solveit’s websocket is down. The user’s websocket is down. Solveit is reacting to the wrong websocket events.

## Important environment detail

Inside Solveit, relative HTTP requests like:

```python
Span(hx_get='/foo', hx_target='closest div')('private')
```

can work because the Solveit backend proxies unhandled HTTP requests to `localhost:8000`.

But that proxy does **not** forward websocket upgrades. So this will not work reliably:

```html
<div hx-ext="ws" ws-connect="/ws2"></div>
```

because `/ws2` goes through Solveit’s own app/proxy path.

For user websockets, use the public `.solveit.pub` domain mapped to the user server port:

```python
ws_url = f"wss://{json.loads(os.environ['PUBLIC_DOMAINS'])[str(srv.port)]}/ws2"
```

Then inject the websocket initiator using that absolute `wss://...` URL:

```python
await add_html_a(Div(Div(ws_connect=ws_url, hx_ext='ws', id='ws2-div'), hx_swap_oob='afterend:#ephemeral'))
```

The `.solveit.pub` mapping supports websocket upgrade; verified with `101 Switching Protocols`.

## Why CORS is not the main issue

Browser WebSocket connections are not blocked by normal fetch/XHR CORS rules. The browser sends an `Origin` header, but it is up to the server to accept or reject it. If the public `.solveit.pub` tunnel forwards the upgrade correctly, the websocket should connect.

## First workaround: capture and block user websocket events

HTMX’s `htmx.on('event', fn)` attaches the listener to `document.body`, not `document`. This matters.

If we add a capturing-phase event listener on `document`, it runs before Solveit’s bubbling-phase listener on `document.body`.

So we can stop user websocket events before Solveit sees them:

```javascript
document.addEventListener('htmx:wsOpen', e => {
  if (e.target?.id !== 'ws2-div') return;
  window._myWs = e.detail.socketWrapper;
  e.stopImmediatePropagation();
  console.log('non-solveit WS opened');
}, true)

document.addEventListener('htmx:wsClose', e => {
  if (e.target?.id !== 'ws2-div') return;
  e.stopImmediatePropagation();
  console.log('blocked close from my ws');
}, true)

document.addEventListener('htmx:wsError', e => {
  if (e.target?.id !== 'ws2-div') return;
  e.stopImmediatePropagation();
  console.log('blocked error from my ws');
}, true)
```

Key details:

- Use `e.target?.id === 'ws2-div'`, not `socketWrapper` comparison.
- `htmx:wsError` uses a different wrapper shape than `htmx:wsOpen` / `htmx:wsClose`, so wrapper identity comparison is fragile.
- Target-element comparison survives reconnects, since each retry still targets the same websocket initiator element.
- This prevents Solveit’s indicator/listeners from reacting to the user websocket.

This fixes the false red/orange Solveit indicator.

## Remaining issue: HTMX still retries

Stopping propagation does not stop HTMX’s internal websocket retry logic. The retry is scheduled inside the websocket extension’s own `onclose` / `onerror` handlers before or independently of the DOM event propagation.

So even if Solveit’s UI no longer reacts, the console may still show repeated failed websocket reconnect attempts.

For intentional shutdown, the clean solution is:

```javascript
const el = document.getElementById('ws2-div')
if (el) el.remove()
```

Then stop the Python server:

```python
stop_server()
```

Removing the websocket initiator element lets HTMX clean up the websocket and avoids the retry loop. In testing, this also avoided triggering the close/error blockers at all.

So the practical shutdown sequence is:

```python
await iife_a(r'''
const el = document.getElementById('ws2-div')
if (el) el.remove()
''')

stop_server()
```

## Best overall pattern

Use both:

1. Capturing listeners as a safety net for unexpected disconnects
2. Remove the websocket initiator element before intentional server shutdown

The listeners prevent Solveit’s UI from reacting to user websocket failures. Removing the initiator avoids HTMX’s retry loop when shutdown is intentional.

## Browser → Python websocket messages

The FastHTML websocket handler expects structured JSON matching the handler parameters. Sending plain text caused a server parse error:

```javascript
window._myWs.send('world')
```

That produced:

```text
Expecting value: line 1 column 1 (char 0)
```

Correct browser send:

```javascript
window._myWs.send(JSON.stringify({msg: 'world'}))
```

Corresponding Python handler:

```python
@app.ws('/ws2', conn=on_connect, disconn=on_disconnect)
async def ws2(msg:str, send):
    _msgs.append(msg)
    if msg == '__init__': return
    await send(Div('Hello ' + msg, id='note2'))
```

## Python → browser websocket messages

FastHTML gives a `send` callable on websocket connect. Store it, then use it later from any async Python cell:

```python
_msgs = []
_ws_send = None

async def on_connect(send):
    global _ws_send
    _ws_send = send
    _msgs.append('Connected!')

async def on_disconnect(ws):
    global _ws_send
    _ws_send = None
    _msgs.append('Disconnected!')
```

Then later:

```python
await _ws_send(Div('Hello from Python', id='note2'))
```

That sends server → browser over the already-open websocket.

## Subtle timing issue with `on_connect`

Sending UI HTML from `on_connect` is unreliable.

Observed behavior:

- Server sent: `<div id="note2">Hello, you have connected</div>`
- Browser websocket inspector showed the message was received
- But HTMX did not swap it into the DOM
- Later messages, sent after explicit browser messages, swapped correctly

Adding `hx-swap-oob="true"` did not fix the first-message issue. A real `setTimeout` delay after `htmx:wsOpen` did work, suggesting a timing/settling race: the websocket is open and messages arrive, but HTMX/Solveit may not yet be ready to process the first swap.

Best practice: treat `conn=on_connect` as lifecycle bookkeeping only. Don’t send UI updates from it.

Good shape:

```python
async def on_connect(send):
    global _ws_send
    _ws_send = send
    _msgs.append('Connected!')

async def on_disconnect(ws):
    global _ws_send
    _ws_send = None
    _msgs.append('Disconnected!')

@app.ws('/ws2', conn=on_connect, disconn=on_disconnect)
async def ws2(msg:str, send):
    _msgs.append(msg)
    if msg == '__init__': return
    await send(Div('Hello ' + msg, id='note2'))
```

## Minimal complete browser-side workaround

```javascript
document.addEventListener('htmx:wsOpen', e => {
  if (e.target?.id !== 'ws2-div') return;
  window._myWs = e.detail.socketWrapper;
  e.stopImmediatePropagation();
  console.log('non-solveit WS opened');
}, true)

document.addEventListener('htmx:wsClose', e => {
  if (e.target?.id !== 'ws2-div') return;
  e.stopImmediatePropagation();
  console.log('blocked close from my ws');
}, true)

document.addEventListener('htmx:wsError', e => {
  if (e.target?.id !== 'ws2-div') return;
  e.stopImmediatePropagation();
  console.log('blocked error from my ws');
}, true)
```

Clean shutdown:

```javascript
const el = document.getElementById('ws2-div')
if (el) el.remove()
```

Browser send:

```javascript
window._myWs.send(JSON.stringify({msg: 'world'}))
```

Python send:

```python
await _ws_send(Div('Hello from Python', id='note2'))
```

## Bottom line

The Solveit bug/oversight is that its websocket monitor listens to all HTMX websocket events globally and assumes they belong to Solveit. User HTMX websockets produce the same events, so user websocket failures are misinterpreted as Solveit connection failures.

The workaround is to:

- Connect user websockets via the public `wss://...solveit.pub/...` URL, not relative paths
- Give the user websocket initiator a stable id, e.g. `ws2-div`
- Add capturing-phase document listeners to block `htmx:wsOpen`, `htmx:wsClose`, and `htmx:wsError` from that element
- Remove the websocket initiator before intentionally stopping the user server
- Avoid sending DOM updates from `on_connect`; send only after the websocket is fully settled or after an explicit client message

## version 0

###### start-section

In [ ]:
# app, rt, srv, p = ensure_server(app=FastHTML(exts='ws', debug=True, 
#     surreal=False, htmx=False, default_hdrs=False))
# 
# _msgs = []
# _ws_send = None
# 
# async def on_connect(send):
#     global _ws_send
#     _ws_send = send
#     print(send)
#     _msgs.append('Connected!')
# 
# async def on_disconnect(ws):
#     global _ws_send
#     _ws_send = None
#     print(ws, dir(ws), ws.send)
#     _msgs.append('Disconnected!')
# 
# @app.ws('/ws2', conn=on_connect, disconn=on_disconnect)
# async def ws2(msg:str, send):
#     _msgs.append(msg)
#     if msg == '__init__': return
#     await send(Div('Hello ' + msg, id="note2"))

In [ ]:
# await iife_a('''
# document.addEventListener('htmx:wsOpen', e => {
#   if (e.target?.id !== 'ws2-div') return;
#   window._myWs = e.detail.socketWrapper;
#   e.stopImmediatePropagation();
#   console.log('blocked open from my ws');
#   setTimeout(() => window._myWs.send(JSON.stringify({msg: '__init__'})), 100)
# }, true)
# 
# document.addEventListener('htmx:wsClose', (e) => {
#   if (e.target?.id === 'ws2-div') { e.stopImmediatePropagation(); console.log('blocked close from my ws') }
# }, true)
# 
# document.addEventListener('htmx:wsError', (e) => {
#   if (e.target?.id === 'ws2-div') { e.stopImmediatePropagation(); console.log('blocked error from my ws') }
# }, true)
# 
# htmx.on('htmx:wsClose', () => console.log('solveit saw wsClose'))
# htmx.on('htmx:wsError', () => console.log('solveit saw wsError'))
# ''')

In [ ]:
# ws_url = f"wss://{json.loads(os.environ['PUBLIC_DOMAINS'])[str(srv.port)]}/ws2"
# await add_html_a(Div(Div(ws_connect=ws_url, hx_ext='ws', id='ws2-div'), hx_swap_oob='afterend:#ephemeral'))

In [ ]:
# while '__init__' not in _msgs: await asyncio.sleep(0.05)
# _msgs

```javascript
window._myWs.send(JSON.stringify({msg: 'world'}))
```

In [ ]:
# await _ws_send(Div('Hello from Python', id='note2'))

In [ ]:
# _msgs

In [ ]:
# res = await js_eval_a('''
# const el = document.getElementById('ws2-div')
# if (el) { el.remove(); return 'removed'; } else { return 'not found'; }
# ''')
# print(res)
# await stop_server_a()

In [ ]:
_user_ws_js = r'''

'''
#| linkedto: _5e8e7990

###### end-section

In [ ]:
_ = '\n'.join((await read_msg()).content.splitlines()[1:])
await link_msg(f"_user_ws_js = r'''{_}'''", msg_type='code', i_collapsed=True);

In [ ]:
_user_ws_js = r'''if (window.user_ws) {
  window.user_ws.listen(false);
  delete window.user_ws;
}

const ids = new Set();

// function _match(e) { return ids.has(e.target?.id) }
function _updateWSIndicator(id, type) {
  const color = { error: "orange", open: "limegreen", close: "red" }[type] || 'grey';
  $(`#${id}-uws-indicator`).css('backgroundColor', color)
}
function _onopen(e) {
  if (!e.target.hasAttribute('data-uws')) return;
  e.target._ws = e.detail.socketWrapper;
  ids.add(e.target.id);
  setTimeout(() => send(e.target.id, {msg: '__init__'}), 250);
  _updateWSIndicator(e.target.id, e.detail.event.type);
  e.stopImmediatePropagation();
}
function _onerr(e) {
  if (!e.target.hasAttribute('data-uws')) return;
  _updateWSIndicator(e.target.id, e.detail.event.type);
  e.stopImmediatePropagation();
}
function listen(on=true) {
  const l = (on?document.addEventListener:document.removeEventListener).bind(document);
  l('htmx:wsOpen', _onopen, true);
  l('htmx:wsClose', _onerr, true);
  l('htmx:wsError', _onerr, true);
}

function cleanup(id) {
  const el = document.getElementById(id);
  if (el) {
    el.remove();
    ids.delete(id);
    _updateWSIndicator(id);
    return 'removed';
  } else { return 'not found'; }
}

function send(id, data) {
  const el = document.getElementById(id);
  if (el?._ws) el._ws.send(typeof data === 'string' ? data : JSON.stringify(data), el);
}

listen();
const user_ws =  { listen, cleanup, send }
window.user_ws = user_ws
export { user_ws }'''
#| linkedto: f2d7c7c4

In [ ]:
async def setup_user_ws(src, nm=''):
    nm = nm or _new_id('user_ws')
    add_mod(nm, src)
    await js_eval_a(f'''await import("/esm/{nm}")''')

In [ ]:
_ar = [Span('⬆︎', style='color:green'), Span('⬇︎', style='color:red')]
def _note(wid, msg, up=False): return Div(_ar[up], f"{wid}{': ' if wid else ''}{msg}", id='uws-note')

In [ ]:
await add_html_a(
    Div(hx_swap_oob='beforebegin:#dialog-container')(
        Div(id='uws-box', cls='flex flex-wrap w-full gap-1 lg:flex-nowrap')(
            _note('', 'Waiting...')
        )));

In [ ]:
def ws_url(port=8000, path='/ws2'):
    return f"wss://{json.loads(os.environ['PUBLIC_DOMAINS'])[str(port)]}{path}"

In [ ]:
def _indicator(wid): return Span(id=f"{wid}-uws-indicator", cls='size-4 border rounded-full min-w-4', title=wid)

In [ ]:
async def user_ws_cleanup(wid):
    res = await js_eval_a(f'''setTimeout(() => {{ $("#{wid}-uws-indicator").remove() }}, 500); return user_ws.cleanup({json.dumps(wid)})''')
    assert res.get('result') in ('removed', 'not found')

In [ ]:
async def user_ws_create_a(wid, wsurl):
    await add_html_a((
        Div(hx_swap_oob='afterend:#ws-div')(Div(ws_connect=wsurl, hx_ext='ws', id=wid, data_uws=True)),
        Div(hx_swap_oob='beforebegin:#uws-note')(_indicator(wid))))

def user_ws_create(wid, wsurl): return asyncio.create_task(user_ws_create_a(wid, wsurl))

async def user_ws_connect(wid, wsurl):
    await user_ws_create_a(wid, wsurl)
    await wait_for_a(lambda: _ws_send is not None)
    await wait_for_a(lambda: (wid, '__init__') in _msgs)

In [ ]:
ws_url(path='/ws2?abc=ABC')

'wss://known-vent-santa-eras34.solveit.pub/ws2?abc=ABC'

In [ ]:
srv, app, rt, p = ensure_server(app=FastHTML(exts='ws', debug=True))#, log_level='info')
await setup_user_ws(_user_ws_js)

In [ ]:
await wait_for_a(lambda: hx.eval('!!window.user_ws'))

In [ ]:
_msgs = []
_ws_send = None

async def on_connect(scope,send):
    global _ws_send
    _ws_send = send
    _msgs.append('Connected!')

async def on_disconnect(scope,ws):
    global _ws_send
    _ws_send = None
    _msgs.append('Disconnected!')

@app.ws('/ws2', conn=on_connect, disconn=on_disconnect)
async def ws2(wid:str, msg:str, send):
    _msgs.append((wid, msg))
    if msg == '__init__': return
    await send(_note(wid, msg))

In [ ]:
wid = 'ws2-div'
res = await js_eval_a(f"return user_ws.cleanup('{wid}')")
test_eq(res['result'], 'not found')

user_ws_create(wid, ws_url(path=f"/ws2?wid={wid}"))
await wait_for_a(lambda: (wid, '__init__') in _msgs)
_msgs

['Connected!', ('ws2-div', '__init__')]

Console Websockets mesages:
```
{"msg":"__init__"}
```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great — that means the good path is working now:

- `htmx:wsOpen` fired
- our capture listener matched `ws2-div`
- it blocked Solveit from seeing the event
- `el._ws` was set
- the delayed `__init__` was sent through the HTMX socket
- Python received the structured message

Next smallest check is to verify Python saw it:

```python
_msgs
```

```json {.usage}
{"summary": "$0.0071", "detail": "total=46,660 | in=46,567 | out=93 | cached=14.0% | $0.0071 | gpt-5.5"}
```

In [ ]:
await iife_a(f"user_ws.send('{wid}', {{msg: 'world'}})")
await wait_for_a(lambda: 'world' in str(_msgs[-1]))

In [ ]:
await user_ws_cleanup(wid)
_msgs

['Connected!', ('ws2-div', '__init__'), ('ws2-div', 'world')]

In [ ]:
_msgs.clear()
wid = 'ws3-div'
await user_ws_connect(wid, ws_url(path=f"/ws2?wid={wid}"))
await _ws_send(_note(wid, 'Hello from Python'))
_msgs

['Connected!', ('ws3-div', '__init__')]

websockets messages:
```
{"msg":"__init__"}	18	11:49:57.800
<div id="note2">Hello from Python</div>	40	11:59:57.474
```
I see `Hello from Python` output.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great — that verifies the full round trip:

- browser → Python works through `user_ws.send`
- Python receives structured JSON as `msg`
- Python → browser works through `_ws_send`
- HTMX swaps the returned `<div id="note2">...</div>` into the page

The next useful refactor target is to stop relying on a single global `_ws_send`, since multiple websocket initiators can connect to the same endpoint. A small registry keyed by websocket id / route will be more reusable.

```json {.usage}
{"summary": "$0.0072", "detail": "total=47,501 | in=47,394 | out=107 | cached=13.8% | $0.0072 | gpt-5.5"}
```

In [ ]:
await user_ws_cleanup('ws3-div')

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Next small step: replace the single `_ws_send` with a tiny registry, while keeping the current `/ws2` handler shape.

```python
_ws_sends = {}

def ws_conn(id):
    async def _conn(send): _ws_sends[id] = send
    return _conn

def ws_disconn(id):
    async def _disconn(ws): _ws_sends.pop(id, None)
    return _disconn
```

Then a tiny helper for Python → browser sends:

```python
async def ws_send(id, o): return await _ws_sends[id](o)
```

```json {.usage}
{"summary": "$0.0072", "detail": "total=47,665 | in=47,537 | out=128 | cached=13.7% | $0.0072 | gpt-5.5"}
```

In [ ]:
_ws_sends = {}

def ws_conn(wid):
    async def _conn(send): _ws_sends[wid] = send
    return _conn

def ws_disconn(wid):
    async def _disconn(ws): _ws_sends.pop(wid, None)
    return _disconn

In [ ]:
async def user_ws_connect(wid, wsurl):
    await user_ws_create_a(wid, wsurl)
    await wait_for_a(lambda: wid in _ws_sends)
    await wait_for_a(lambda: (wid, '__init__') in _msgs)

In [ ]:
async def ws_send(wid, o): return await _ws_sends[wid](o)

In [ ]:
wid = 'ws4-div'
@app.ws('/ws4', conn=ws_conn(wid), disconn=ws_disconn(wid))
async def ws4(wid:str, msg:str, send):
    _msgs.append((wid, msg))
    if msg == '__init__': return
    await send(_note(wid, msg))

In [ ]:
await user_ws_connect(wid, ws_url(path=f"/ws4?wid={wid}"))
_ws_sends.keys()

dict_keys(['ws4-div'])

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good. Now verify browser → Python and Python → browser on the registry-backed socket:

```python
await iife_a("user_ws.send('ws4-div', {msg: 'registry'})")
await wait_for_a(lambda: 'registry' in _msgs)
await ws_send('ws4-div', Div('Hello from registry', id='note2'))
```

```json {.usage}
{"summary": "$0.0073", "detail": "total=48,380 | in=48,303 | out=77 | cached=13.5% | $0.0073 | gpt-5.5"}
```

In [ ]:
await iife_a(f"user_ws.send('{wid}', {{msg: 'registry'}})")
await wait_for_a(lambda: 'registry' in str(_msgs[-1]))
await ws_send(wid, _note(wid, 'Hello from registry'))

```
{"msg":"registry"}	18	12:03:38.200
{"msg":"__init____"}	18	12:04:06.446
<div id="note2">Hello registry</div>	37	12:04:06.562
<div id="note2">Hello from registry</div>	42	12:04:06.571
```

`#note2` updated.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Excellent — the registry-backed version is verified in both directions.

Next small refactor: make a helper that both registers the route and returns the initiator id/path pair, so callers don’t have to repeat the id in three places.

```python
def add_user_ws(app, id, path, handler, target='note2'):
    app.ws(path, conn=ws_conn(id), disconn=ws_disconn(id))(handler)
    return id,path
```

A tiny test endpoint can use it like this:

```python
async def ws5(msg:str, send):
    _msgs.append(msg)
    if msg == '__init__': return
    await send(Div('Hello ' + msg, id='note2'))

add_user_ws(app, 'ws5-div', '/ws5', ws5)
```

```json {.usage}
{"summary": "$0.0074", "detail": "total=48,792 | in=48,621 | out=171 | cached=5.0% | $0.0074 | gpt-5.5"}
```

In [ ]:
def add_user_ws(app, wid, path, handler):
    app.ws(path, conn=ws_conn(wid), disconn=ws_disconn(wid))(handler)

In [ ]:
async def ws5(wid:str, msg:str, send):
    _msgs.append((wid, msg))
    if msg == '__init__': return
    await send(_note(wid, msg))

wid, path = 'ws5-div', '/ws5'
add_user_ws(app, wid, path, ws5)

In [ ]:
await user_ws_connect(wid, ws_url(path=path+f"?wid={wid}"))
await iife_a(f"user_ws.send('{wid}', {{msg: 'helper'}})")
await wait_for_a(lambda: 'helper' in str(_msgs[-1]))
await ws_send(wid, _note(wid, 'Hello from helper'))

In [ ]:
async def start_user_ws(srv, wid, path): return await user_ws_connect(wid, ws_url(srv.port, path+f"?wid={wid}"))

In [ ]:
wid, path = 'ws6-div', '/ws6'
add_user_ws(app, wid, path, ws5)
await start_user_ws(srv, wid, path)
await iife_a(f"user_ws.send('{wid}', {{msg: 'started'}})")
await wait_for_a(lambda: 'started' in str(_msgs[-1]))
await ws_send(wid, _note(wid, 'Hello from start_user_ws'))

All good.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Nice. We now have the pieces separated cleanly:

- `ws_url` builds the public websocket URL
- `user_ws_create` creates/processes the browser initiator
- `_ws_sends` tracks active Python send callables by initiator id
- `ws_conn` / `ws_disconn` manage that registry
- `ws_send` sends Python → browser
- `add_user_ws` registers an endpoint
- `start_user_ws` starts the browser side

The next worthwhile step is packaging these into a small reusable section, probably with the JS string plus these Python helpers together.

```json {.usage}
{"summary": "$0.0076", "detail": "total=50,013 | in=49,886 | out=127 | cached=95.2% | $0.0076 | gpt-5.5"}
```

In [ ]:
await user_ws_cleanup('ws4-div')
await user_ws_cleanup('ws5-div')
await user_ws_cleanup('ws6-div');

## version 1

Let's consolidate the Python-facing helper layer first:

In [ ]:
def user_ws_url(srv, path, qs=''): return f"wss://{json.loads(os.environ['PUBLIC_DOMAINS'])[str(srv.port)]}{path}{qs}"

async def user_ws_call(fn, *args):
    args = ', '.join(json.dumps(o) for o in args)
    return await js_eval_a(f"return user_ws.{fn}({args})")

async def user_ws_send(wid, data):
    l = len(_msgs)
    await iife_a(f"user_ws.send({json.dumps(wid)}, {json.dumps(data)})")
    await wait_for_a(lambda: len(_msgs)>l)

In [ ]:
wid,path = 'ws7-div', '/ws7'
add_user_ws(app, wid, path, ws5)
await user_ws_connect(wid, user_ws_url(srv, path, f"?wid={wid}"))
await user_ws_send(wid, dict(msg='packed'))
test_is('packed' in str(_msgs[-1]), True)
await ws_send(wid, _note(wid, 'Hello from packed helpers'))

In [ ]:
await user_ws_cleanup(wid)

Consolidate the server-side registry helpers into the same reusable shape, with a clearer public name:

In [ ]:
user_ws_sends = {}

def user_ws_conn(wid):
    async def _conn(send): user_ws_sends[wid] = send
    return _conn

def user_ws_disconn(wid):
    async def _disconn(ws): user_ws_sends.pop(wid, None)
    return _disconn

async def user_ws_send_py(wid, o): return await user_ws_sends[wid](o)

def add_user_ws(app, wid, path, handler):
    app.ws(path, conn=user_ws_conn(wid), disconn=user_ws_disconn(wid))(handler)

async def user_ws_connect(wid, wsurl):
    await user_ws_create_a(wid, wsurl)
    await wait_for_a(lambda: wid in user_ws_sends)
    await wait_for_a(lambda: (wid, '__init__') in _msgs)

In [ ]:
wid,path = 'ws8-div', '/ws8'
add_user_ws(app, wid, path, ws5)
await user_ws_connect(wid, user_ws_url(srv, path, f"?wid={wid}"))
await user_ws_send(wid, dict(msg='renamed'))
await user_ws_send_py(wid, _note(wid, 'Hello from renamed registry'))
await user_ws_cleanup(wid)

Make a single convenience function that registers the route and starts the browser initiator.

In [ ]:
async def start_user_ws(app, srv, wid, path, handler):
    add_user_ws(app, wid, path, handler)
    return await user_ws_connect(wid, user_ws_url(srv, path, f"?wid={wid}"))

In [ ]:
wid, path = 'ws9-div', '/ws9'
await start_user_ws(app, srv, wid, path, ws5)
await user_ws_send(wid, dict(msg='onecall'))
test_is('onecall' in str(_msgs[-1]), True)
await user_ws_send_py(wid, _note(wid, 'Hello from start_user_ws'))

Make shutdown symmetric so intentional shutdown removes the initiator first and then drops the stored send callable.

In [ ]:
async def stop_user_ws(id):
    try: await user_ws_cleanup(id)
    finally: user_ws_sends.pop(id, None)
    assert wid not in user_ws_sends

In [ ]:
await stop_user_ws(wid)

That's a lot and a lot of `user_ws_` prefixes.  Time to put in a class, don't you think?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — this is the right point to put the public surface behind one object. The class can own `app`, `srv`, JS namespace, and the active send registry.

In [ ]:
class UserWS:
    def __init__(self, app, srv, js='user_ws'): store_attr(); self.sends, self._msgs = {}, []

    def url(self, path, qs=''): return f"wss://{json.loads(os.environ['PUBLIC_DOMAINS'])[str(self.srv.port)]}{path}{qs}"

    async def create(self, wid, path):
        wsurl = path if path.startswith('ws') else self.url(path, f"?wid={wid}")
        await add_html_a((
            Div(hx_swap_oob='afterend:#ws-div')(Div(ws_connect=wsurl, hx_ext='ws', id=wid, data_uws=True)),
            Div(hx_swap_oob='beforebegin:#uws-note')(_indicator(wid))))

    async def cleanup(self, wid):
        res = await js_eval_a(f'''setTimeout(() => {{ $("#{wid}-uws-indicator").remove() }}, 500); return user_ws.cleanup({json.dumps(wid)})''')
        assert res.get('result') in ('removed', 'not found')
    async def send_client(self, wid, data):
        l = len(self._msgs)
        await iife_a(f"{self.js}.send({json.dumps(wid)}, {json.dumps(data)})")
        await wait_for_a(lambda: len(self._msgs)>l)
    async def send(self, wid, o):
        assert wid in self.sends, f"{wid} is not connected"
        return await self.sends[wid](o)

    def conn(self, wid):
        async def _conn(send): self.sends[wid] = send
        return _conn
    def disconn(self, wid):
        async def _disconn(ws): self.sends.pop(wid, None)
        return _disconn

    def add(self, path, handler, wid=''):
        wid = wid or _new_id('uws')
        self.app.ws(path, conn=self.conn(wid), disconn=self.disconn(wid))(handler)
        return wid
    async def connect(self, wid, path):
        await self.create(wid, path)
        await wait_for_a(lambda: wid in self.sends)
        await wait_for_a(lambda: (wid, '__init__') in self._msgs)
    async def start(self, path, handler, wid=''):
        wid = self.add(path, handler, wid=wid)
        await self.connect(wid, path)
        return wid
    async def restart(self, path, handler, wid): return await self.start(path, handler, wid=wid)

    async def stop(self, wid):
        try: await self.cleanup(wid)
        finally: self.sends.pop(wid, None)
        assert wid not in self.sends

In [ ]:
uws = UserWS(app, srv)
_msgs = uws._msgs

In [ ]:
wid = await uws.start(path := '/ws12', ws5)
wid

'uws_2'

In [ ]:
await uws.send_client(wid, dict(msg='classy'))
test_is('classy' in str(_msgs[-1]), True)
await uws.send(wid, _note(wid, 'Hello from UserWS'))
uws._msgs

[('uws_2', '__init__'), ('uws_2', 'classy')]

All good.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great. The class shape is now verified:

- `uws.start(...)` registers the endpoint, creates/processes the browser initiator, and waits for connection
- `uws.send_client(...)` sends browser → Python
- `uws.send(...)` sends Python → browser
- `uws.sends` cleanly scopes active send callables per `UserWS` instance

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Next useful cleanup: make `start` return `wid` as before, but first clean up any existing initiator with the same id to avoid duplicate DOM elements/retry loops during iteration.

In [ ]:
@patch
async def start(self:UserWS, path, handler, wid=''):
    if wid: await self.stop(wid)
    wid = self.add(path, handler, wid=wid)
    await self.connect(wid, path)
    return wid

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

One thing to be careful about: `FastHTML.app.ws(...)` may not like re-registering the same route path during iteration. Cleaning up the DOM initiator is good, but `self.add(...)` still registers the route again.

So next tiny refinement: split “route already added” from “browser initiator started” by tracking registered websocket ids.

In [ ]:
@patch
def _init_routes(self:UserWS):
    if not hasattr(self, 'routes'): self.routes = set()

@patch
def add(self:UserWS, path, handler, wid=''):
    self._init_routes()
    wid = wid or _new_id('uws')
    if wid not in self.routes:
        self.app.ws(path, conn=self.conn(wid), disconn=self.disconn(wid))(handler)
        self.routes.add(wid)
    return wid

In [ ]:
await uws.start(path, ws5, wid)
test_eq((wid in uws.sends, wid in uws.routes), (True, True))

In [ ]:
await uws.send_client(wid, dict(msg='restarted'))
await uws.send(wid, _note(wid, 'Hello after restart'))
await uws.stop(wid)

All good.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Nice. One more small polish: initialize `routes` in `__init__` so `_init_routes` isn't needed in the final class.

In [ ]:
@patch
def __init__(self:UserWS, app, srv, js='user_ws'):
    store_attr()
    self.sends,self.routes,self._msgs = {},set(),[]

@patch
def add(self:UserWS, path, handler, wid=''):
    wid = wid or _new_id('uws')
    if wid not in self.routes:
        self.app.ws(path, conn=self.conn(wid), disconn=self.disconn(wid))(handler)
        self.routes.add(wid)
    return wid

In [ ]:
uws2 = UserWS(app, srv)
_msgs = uws2._msgs
wid2 = await uws2.start(path2 := '/ws13', ws5, wid)
test_eq((wid2 == wid, wid2 in uws2.sends, wid2 in uws2.routes), (True, True, True))

In [ ]:
await uws2.send_client(wid2, dict(msg='fresh_instance'))
test_is('fresh_instance' in str(_msgs[-1]), True)
await uws2.send(wid2, _note(wid, 'Hello from fresh instance'))

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Next small tightening: `routes` should probably be a dict from `wid → path`, not a set. That makes restart idempotent for the same `wid/path`, but catches accidental reuse of the same `wid` for a different route.

```python
@patch
def __init__(self:UserWS, app, srv, js='user_ws'):
    store_attr()
    self.sends,self.routes = {},{}

@patch
def add(self:UserWS, path, handler, wid=''):
    wid = wid or _new_id('uws')
    if wid in self.routes:
        assert self.routes[wid] == path, f"{wid} already registered for {self.routes[wid]}, not {path}"
        return wid
    self.app.ws(path, conn=self.conn(wid), disconn=self.disconn(wid))(handler)
    self.routes[wid] = path
    return wid
```

```json {.usage}
{"summary": "$0.0074", "detail": "total=46,295 | in=45,616 | out=679 | cached=14.3% | reasoning=483 | $0.0074 | gpt-5.5"}
```

In [ ]:
@patch
def __init__(self:UserWS, app, srv, js='user_ws'):
    store_attr()
    self.sends,self.routes,self._msgs = {},{},[]

@patch
def add(self:UserWS, path, handler, wid=''):
    wid = wid or _new_id('uws')
    if wid in self.routes:
        assert self.routes[wid] == path, f"{wid} already registered for {self.routes[wid]}, not {path}"
        return wid
    self.app.ws(path, conn=self.conn(wid), disconn=self.disconn(wid))(handler)
    self.routes[wid] = path
    return wid

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Tiny test for the duplicate-`wid` guard:

```python
uws3 = UserWS(app, srv)
wid3 = uws3.add('/ws14', ws5, 'same-wid')
test_eq(uws3.add('/ws14', ws5, 'same-wid'), wid3)
test_fail(lambda: uws3.add('/ws15', ws5, 'same-wid'), contains='already registered')
```

```json {.usage}
{"summary": "$0.0070", "detail": "total=46,115 | in=46,020 | out=95 | cached=5.3% | $0.0070 | gpt-5.5"}
```

In [ ]:
uws3 = UserWS(app, srv)
_msgs = uws3._msgs
wid3 = uws3.add('/ws14', ws5, 'same-wid')
test_eq(uws3.add('/ws14', ws5, 'same-wid'), wid3)
test_fail(lambda: uws3.add('/ws15', ws5, 'same-wid'), contains='already registered')

In [ ]:
wid4 = await uws3.start('/ws14', ws5, 'same-wid')
res = await uws3.stop(wid4)
test_is(wid4 in uws3.sends, False)
await uws3.stop('missing-wid')

In [ ]:
await uws2.stop(wid2)
await stop_server_a()

## version 2

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good. The class works from a fresh instance.

Next, let's prepare the final consolidated JS string and class in two clean cells. First the JS string:

In [ ]:
_ = '\n'.join((await read_msg()).content.splitlines()[1:])
await link_msg(f"_user_ws_js = r'''{_}'''", msg_type='code', i_collapsed=True, exported=True);

In [ ]:
_user_ws_js = r'''if (window.user_ws) {
  window.user_ws.listen(false);
  delete window.user_ws;
}

function hooks() { return {open:[], close:[], err:[], cleanup:[]} }
const _hooks = hooks()
function on(name, fn) { _hooks[name].push(fn); return () => off(name, fn); }
function off(name, fn) { _hooks[name] = _hooks[name].filter(o => o !== fn) }
function call(name, ...args) { for (const fn of _hooks[name]) fn(...args); }

function _elt(e) { return e.detail?.elt || e.target }
function _onopen(e) {
  const elt = _elt(e);
  if (!elt?.hasAttribute('data-uws')) return;
  e.stopImmediatePropagation();
  elt._ws = e.detail.socketWrapper;
  setTimeout(() => send(elt.id, {event:'__init__'}), 250);
  call('open', elt, e);
}
function _onerr(e) {
  const elt = _elt(e);
  if (!elt?.hasAttribute('data-uws')) return;
  e.stopImmediatePropagation();
  call(e.type === 'htmx:wsClose' ? 'close' : 'err', elt, e)
}
async function _opened(e) {
  const {idx,wid,timeout=3000} = e.detail;
  const end = Date.now()+timeout;
  while (Date.now()<end) {
    const el = document.getElementById(wid);
    if (el?._ws) return pushData(idx, {open:true});
    await new Promise(r => setTimeout(r, 25));
  }
  pushData(idx, {open:false});
}
function listen(on=true) {
  const l = (on ? document.addEventListener : document.removeEventListener).bind(document);
  l('htmx:wsOpen', _onopen, true);
  l('htmx:wsClose', _onerr, true);
  l('htmx:wsError', _onerr, true);
  l('uws:open?', _opened, false);
}
function cleanup(id) {
  const el = document.getElementById(id);
  if (el) {
    el.remove();
    call('cleanup', el);
    return 'removed';
  } else return 'not found';
}
function send(id, data={}) {
  const el = document.getElementById(id);
  if (!el?._ws) return;
  const o = typeof data === 'string' ? {msg:data} : {...data};
  o.wid ??= id;
  el._ws.send(JSON.stringify(o), el);
}

listen();
const doc = 'Browser helper for Solveit-safe HTMX websocket initiators';
const user_ws = {doc, listen, cleanup, send, on, off};
window.user_ws = user_ws
export { user_ws }'''
#| linkedto: 99847640

In [ ]:
_ = '\n'.join((await read_msg()).content.splitlines()[1:])
await link_msg(f"_user_ws_indicator_js = r'''{_}'''", msg_type='code', i_collapsed=True, exported=True);

In [ ]:
_user_ws_indicator_js = r'''import { user_ws } from '/esm/user_ws'

function truncS(s, l=120) { s = String(s); return s.length <= l ? s : s.substring(0, l) + '\u2026'; }
function shownote(id, msg, dir, color) {
  $('#uws-note').empty()
    .append($('<span>').css('color', color).text(dir))
    .append(document.createTextNode(`${id}: `))
    .append($('<samp>').text(truncS(msg)))
}
function noteup(e) { shownote(e.detail.elt.id, e.detail.message, '⬆︎', 'green') }
function notedown(e) { shownote(e.detail.elt.id, e.detail.message, '⬇︎', 'red') }
function onopen(el,e) {
  $(`#${el.id}-uws-indicator`).css('backgroundColor', 'limegreen');
  htmx.on(el, 'htmx:wsAfterMessage', noteup);
  htmx.on(el, 'htmx:wsAfterSend', notedown);
  shownote(el.id, 'opened', '⬇︎', 'red')
}
function onerr(el,e) {
  const type = e.detail?.event?.type || 'error';
  const color = { error: 'orange', close: 'red' }[type] || 'grey';
  $(`#${el.id}-uws-indicator`).css('backgroundColor', color);
}
function oncleanup(el) {
  $(`#${el.id}-uws-indicator`).css('backgroundColor', 'grey');
  htmx.off(el, 'htmx:wsAfterMessage', noteup);
  htmx.off(el, 'htmx:wsAfterSend', notedown);
  shownote(el.id, 'closed', '⬇︎', 'red')
}

if (window._uws_indicator_offs) window._uws_indicator_offs.forEach(f => f());
window._uws_indicator_offs = [
  user_ws.on('open', onopen),
  user_ws.on('close', onerr),
  user_ws.on('err', onerr),
  user_ws.on('cleanup', oncleanup)
];'''
#| linkedto: f6a6c557

In [ ]:
test_eq(await hx.remove('#uws-note'), True)

In [ ]:
#| exporti
async def setup_uws_indicator():
    await iife_a(r"document.getElementById('uws-box')?.remove()")
    await add_html_a(
        Div(hx_swap_oob='beforebegin:#dialog-container')(
            Div(id='uws-box', cls='flex flex-wrap w-full gap-1 lg:flex-nowrap')(
                Div('uws indicator...', id='uws-note')
            )))
    add_mod('user_ws_indicator', _user_ws_indicator_js)
    await js_eval_a('''await import("/esm/user_ws_indicator")''')

In [ ]:
async def setup_user_ws(indicator=False):
    "Inject the browser-side user_ws helper; use only in Solveit environment"
    async with serve_app() as _:
        add_mod('user_ws', _user_ws_js)
        await js_eval_a('''await import("/esm/user_ws")''')
        if indicator:await setup_uws_indicator()

In [ ]:
await setup_user_ws(True)

The browser helper in `_user_ws_js` is injected once with `setup_user_ws()`, while `UserWS(app, srv)` instances can be created many times to manage independent route/send registries.

The JS helper tracks websocket initiator elements, stores each HTMX socket wrapper on its DOM element as `el._ws`, and filters HTMX websocket events by `data-uws` attribute. Matching `htmx:wsOpen`, `htmx:wsClose`, and `htmx:wsError` events are stopped in the capture phase so Solveit’s own global websocket monitor does not mistake user websocket lifecycle events for Solveit backend failures.

In [ ]:
#| exporti
async def noop_a(*args, **kwargs): pass

async def create_uws_indicator(wid):
    return await add_html_a(Div(hx_swap_oob='beforebegin:#uws-note')(_indicator(wid)))

async def cleanup_uws_indicator(wid):
    return await js_eval_a(f'''setTimeout(() => {{ $("#{wid}-uws-indicator").remove() }}, 500); return true''')

uws_ind = dict(create_ui=create_uws_indicator, cleanup_ui=cleanup_uws_indicator)

In [ ]:
class UserWS:
    "Manage Solveit-safe HTMX websocket initiators and server send handles keyed by `wid`"
    def __init__(self, app, srv, create_ui=noop_a, cleanup_ui=noop_a): store_attr(); self.sends,self.routes,self.paths = {},{},{}

    def url(self, path, qs=''):
        "Return public websocket URL for `path`"
        return f"wss://{json.loads(os.environ['PUBLIC_DOMAINS'])[str(self.srv.port)]}{path}{qs}"

    _js='user_ws'
    async def create(self, wid, path, **kwargs):
        "Create browser websocket initiator for `path`"
        qs = f"{'&' if '?' in path else '?'}{urlencode(dict(wid=wid))}"
        wsurl = path if path.startswith(('ws://', 'wss://')) else self.url(path, qs)
        await add_html_a(Div(hx_swap_oob='afterend:#ws-div')(Div(ws_connect=wsurl, hx_ext='ws', id=wid, data_uws=True, **kwargs)))
        await self.create_ui(wid)

    async def cleanup(self, wid):
        "Remove browser initiator element"
        try:
            res = await js_eval_a(f'''return {self._js}.cleanup({json.dumps(wid)})''')
            assert res.get('result') in ('removed', 'not found')
        finally: await self.cleanup_ui(wid)
    async def send_client(self, wid, data):
        "Send browser to Python via JS helper"
        await iife_a(f"{self._js}.send({json.dumps(wid)}, {json.dumps(data)})")
    async def send(self, wid, o):
        "Send Python to browser via stored send callable"
        assert wid in self.sends, f"{wid} is not connected"
        return await self.sends[wid](o)

    def _conn(self, wid, conn):
        "Return FastHTML connect hook storing server send callable"
        async def _conn(scope,ws,send):
            self.sends[wid] = send
            if conn: await conn(scope,ws,send)
        return _conn
    def _disconn(self, wid, disconn):
        "Return FastHTML disconnect hook clearing server send callable"
        async def _disconn(scope,ws):
            self.sends.pop(wid, None)
            if disconn: await disconn(scope,ws)
        return _disconn

    def add(self, path, handler, wid='', conn=None, disconn=None, **kwargs
    ) -> str: # `wid`
        "Register websocket route once; optionally setup conn/disconn hooks"
        wid = wid or _new_id('uws')
        if wid in self.routes:
            assert self.routes[wid] == path, f"{wid} already registered for {self.routes[wid]}, not {path}"
            return wid
        if path in self.paths: raise AssertionError(f"{path} already registered for {self.paths[path]}, not {wid}")
        self.app.ws(path, conn=self._conn(wid, conn), disconn=self._disconn(wid, disconn), **kwargs)(handler)
        self.routes[wid],self.paths[path] = path,wid
        return wid
    async def wait_open(self, wid, timeout=3000):
        res = await event_get_a('uws:open?', wid=wid, timeout=timeout)
        assert res.get('open'), f"{wid} did not open"
    @delegates(add, keep=True)
    async def start(self, path, handler, wid='', initkw=None, **kwargs
    )-> str: # `wid`
        "Register route and create browser initiator"
        if wid: await self.stop(wid)
        wid = self.add(path, handler, wid, **kwargs)
        await self.create(wid, path, **(initkw or {}))
        await self.wait_open(wid)
        await wait_for_a(lambda: wid in self.sends)
        return wid

    async def stop(self, wid):
        "Remove browser initiator and forget server send callable"
        try: await self.cleanup(wid)
        finally: self.sends.pop(wid, None)

In [ ]:
_msgs = []
Div(id=(did := f"_{simple_id()}"))

<div id="_b361b170"></div>

In [ ]:
srv, app, rt, p = ensure_server(app=FastHTML(exts='ws', debug=True))

async def ws_msg(data:dict, ws, send):
    _msgs.append(data)
    if data.get('event') == '__init__': return
    await send(Div(Samp(json.dumps(data)), Br(), id=did, hx_swap_oob="beforeend"))

# async with serve_app(app_kw=AD(exts='ws')) as srv:#, log_level='trace') as srv:
uws3 = UserWS(srv.app, srv, **uws_ind)    
wid = await uws3.start(path := '/ws14', ws_msg)

print(_msgs, '\n----\n')

await uws3.send_client(wid, dict(msg='clean'))
await wait_for_a(lambda: _msgs and 'clean' in str(_msgs[-1]))
await uws3.send(wid, Div(Samp('Hello from cleaned UserWS'), Br(), id=did, hx_swap_oob="beforeend"))

await uws3.stop(wid)
print(_msgs)

[] 
----


[{'event': '__init__', 'wid': 'uws_3'}, {'msg': 'clean', 'wid': 'uws_3'}]


In [ ]:
await stop_server_a()

In [ ]:
_msgs, events = [], []
async def _conn(scope,ws,send): events.append('connected')
async def _disconn(scope,ws): events.append('disconnected')

async with serve_app() as srv:
    uws4 = UserWS(srv.app, srv, **uws_ind)
    wid = await uws4.start(path := '/ws15', ws_msg, conn=_conn, disconn=_disconn)
    await uws4.send_client(wid, dict(msg='doc_ok'))
    await wait_for_a(lambda: _msgs and 'doc_ok' in str(_msgs[-1]))
    await uws4.send(wid, Div(Samp('Hello after doc reinject'), Br(), id=did, hx_swap_oob="beforeend"))
    await uws4.stop(wid)
_msgs, events

([{'event': '__init__', 'wid': 'uws_4'}, {'msg': 'doc_ok', 'wid': 'uws_4'}],
 ['connected', 'disconnected'])

Lifecycle regression test: stopping and restarting the same `wid` must reuse its registered route and restore a live send handle.

In [ ]:
_msgs = []

async with serve_app() as srv:
    uws = UserWS(srv.app, srv, **uws_ind)
    wid = await uws.start('/ws16', ws_msg, wid='restart-test')
    await uws.stop(wid)
    wid = await uws.start('/ws16', ws_msg, wid=wid)
    test_eq((wid, wid in uws.routes, wid in uws.sends), ('restart-test', True, True))
    await uws.stop(wid)

This pins down the intended contract: stop removes the browser connection and send handle, but preserves route registration for a later restart.

Add the complementary route-collision test: one path must not be registered under two distinct websocket ids.

In [ ]:
async with serve_app() as srv:
    uws = UserWS(srv.app, srv)
    uws.add('/ws17', ws_msg, wid='first-wid')
    test_fail(lambda: uws.add('/ws17', ws_msg, wid='second-wid'), contains='already registered')

`send_client` adds `wid` automatically when callers omit it:

In [ ]:
_msgs = []

async with serve_app() as srv:
    uws = UserWS(srv.app, srv)
    wid = await uws.start('/ws18', ws_msg, wid='client-wid')
    await uws.send_client(wid, dict(msg='auto-wid'))
    await wait_for_a(lambda: _msgs and _msgs[-1].get('msg') == 'auto-wid')
    test_eq(_msgs[-1], dict(msg='auto-wid', wid='client-wid'))
    await uws.stop(wid)

In [ ]:
events = []
async def _disconn(scope, ws): events.append('disconnected')

uws = UserWS(None, None)
uws.sends['test-wid'] = object()

await uws._disconn('test-wid', _disconn)(None, None)

test_eq(events, ['disconnected'])
test_eq(uws.sends, {})

The disconnect hook clears the stored handle and then calls the user hook. Test the symmetric connection contract:

In [ ]:
events = []

async def _conn(scope, ws, send): events.append('connected')
async def send(o): pass

uws = UserWS(None, None)
await uws._conn('test-wid', _conn)(None, None, send)

test_eq(events, ['connected'])
test_is(uws.sends['test-wid'], send)

The remaining non-obvious lifecycle contract is ordering: the stored send handle should already be available inside a user-supplied connection hook.

In [ ]:
seen = []

async def _conn(scope, ws, send): seen.append(uws.sends.get('test-wid') is send)
async def send(o): pass

uws = UserWS(None, None)
await uws._conn('test-wid', _conn)(None, None, send)

test_eq(seen, [True])

The connection handle is available before a user connection hook runs. Pin down the reciprocal ordering: it should be removed before a user disconnect hook runs.

In [ ]:
seen = []

async def _disconn(scope, ws): seen.append('test-wid' not in uws.sends)

uws = UserWS(None, None)
uws.sends['test-wid'] = object()
await uws._disconn('test-wid', _disconn)(None, None)

test_eq(seen, [True])

The connection and disconnection ordering contracts are covered.

Version 2 can now shift from lifecycle internals to a concise public usage example: `setup_user_ws()`, construct `UserWS`, `start`, send in both directions, then `stop`.

In [ ]:
await stop_server_a()
srv, app, rt, p = ensure_server()

In [ ]:
await setup_user_ws()

In [ ]:
got = []

async def echo_check(data:dict, ws, send):
    if data.get('event') == '__init__': return
    got.append(data)
    await send(Div(data['msg'], id='uws-reply'))

uws = UserWS(app, srv)

wid = await uws.start('/echo-check', echo_check)
await uws.send_client(wid, 'hello')
await wait_for_a(lambda: got)
test_eq(got, [dict(msg='hello', wid=wid)])
await uws.stop(wid)

In [ ]:
await stop_server_a()

The end-to-end public example passes against a fresh server:

- `setup_user_ws()` installs the browser helper.
- `UserWS.start()` opens the user websocket.
- `send_client(wid, 'hello')` produces `dict(msg='hello', wid=wid)` in the handler.
- `stop(wid)` cleans up the initiator and connection.

The helper’s core lifecycle and public usage contract are now covered.

The hooks system is the remaining gap. It is used by the indicator module, but we have not directly pinned down its public contract.

Start with the normal lifecycle hooks: `open` and `cleanup`.

In [ ]:
await iife_a(r'''
window._uws_events = []
window._uws_offs = [
  user_ws.on('open', el => window._uws_events.push(['open', el.id])),
  user_ws.on('cleanup', el => window._uws_events.push(['cleanup', el.id]))
]
''')

async with serve_app() as srv:
    uws = UserWS(srv.app, srv)
    wid = await uws.start('/ws-hooks', ws_msg, wid='hook-test')
    await uws.stop(wid)

res = await js_eval_a('return window._uws_events')
test_eq(res['result'], [['open', 'hook-test'], ['cleanup', 'hook-test']])

await iife_a('window._uws_offs.forEach(fn => fn()); delete window._uws_offs')

That verifies registration via `on`, dispatch through `call`, and deregistration using the unsubscribe functions returned by `on`.

`close` and `err` still need an intentional-disconnect test. They matter because those hooks drive the red/orange indicator states; the earlier websocket work established event filtering, but not this hook API boundary directly.

The lifecycle hooks passed: `open` fired when the initiator connected, and `cleanup` fired when `stop` removed it.

Test the unsubscribe function returned by `on`:

In [ ]:
await iife_a(r'''
window._uws_events = []
const off = user_ws.on('open', el => window._uws_events.push(el.id))
off()
''')

async with serve_app() as srv:
    uws = UserWS(srv.app, srv)
    wid = await uws.start('/ws-hooks-off', ws_msg, wid='hook-off-test')
    await uws.stop(wid)

res = await js_eval_a('return window._uws_events')
test_eq(res['result'], [])

That verifies that a removed hook cannot fire on a subsequent websocket lifecycle event.

The unsubscribe test passed: a hook removed through the function returned by `on(...)` did not fire.

Test `close` and `err` dispatch directly. This isolates the public hook routing without deliberately breaking a live server:

In [ ]:
res = await js_eval_a(r'''
const el = document.createElement('div')
el.id = 'hook-event-test'
el.setAttribute('data-uws', '')
document.body.append(el)

const events = []
const offs = [
  user_ws.on('close', el => events.push(['close', el.id])),
  user_ws.on('err', el => events.push(['err', el.id]))
]

el.dispatchEvent(new CustomEvent('htmx:wsClose', {bubbles:true, detail:{elt:el, event:{type:'close'}}}))
el.dispatchEvent(new CustomEvent('htmx:wsError', {bubbles:true, detail:{elt:el, event:{type:'error'}}}))

offs.forEach(fn => fn())
el.remove()
return events
''')

test_eq(res['result'], [['close', 'hook-event-test'], ['err', 'hook-event-test']])

That verifies that the capture listeners classify `htmx:wsClose` as `close` and `htmx:wsError` as `err`, then pass the websocket initiator element to each registered hook.

The hook-routing test passed: `htmx:wsClose` dispatches `close`, and `htmx:wsError` dispatches `err`, each with the marked initiator element.

The hook system is now covered for:

- registration and dispatch (`open`, `cleanup`)
- unsubscribe functions returned by `on`
- event classification (`close` versus `err`)
- correct initiator identity passed to callbacks

That closes the remaining Version 2 test gap.

The remaining high-value checks are integration checks, not more unit tests:

- In a fresh kernel, run `setup_user_ws()`, create one `UserWS`, then verify `start`, browser→Python, Python→browser, and `stop`.
- Force an unexpected user-websocket disconnect and confirm Solveit’s own connection indicator and submit/cancel controls remain unchanged. That is the original isolation contract.
- Confirm two `UserWS` instances can use distinct routes and `wid`s without their send registries crossing.

The registry and lifecycle-wrapper contracts are already covered directly.

`UserWS` provides Solveit-safe user HTMX websockets. One `UserWS` instance can manage many independent websocket initiators, keyed by `wid`.

Typical flow:

1. `await setup_user_ws()` once per browser session
2. `uws = UserWS(app, srv)`
3. define a FastHTML websocket handler
4. `wid = await uws.start('/ws', handler)`
5. browser → Python: `await uws.send_client(wid, dict(msg='hi'))`
6. Python → browser: `await uws.send(wid, Div('hello', id='target'))`
7. cleanup before stopping the server: `await uws.stop(wid)`

`UserWS.start()` removes any existing browser initiator with the same supplied `wid` before recreating it, avoiding duplicate DOM elements and HTMX retry loops during iteration. Routes are registered only once per `wid` per `UserWS` instance.

Python handlers passed to `UserWS.start()` are normal FastHTML websocket handlers registered with `app.ws(...)`. They take the usual parsed parameters plus special params like `send`, and can send normal FastHTML/HTML responses. `UserWS` only manages connection setup, event isolation, and access to the server-side `send` callable; it does not change FastHTML websocket handler semantics.

### Environment notes

- Solveit's HTTP proxy does not forward websocket upgrades; always use the public `wss://...solveit.pub/...` URL from `UserWS.url()`, not relative paths.
- `on_connect` has a timing race: the socket is open and messages arrive, but HTMX may not yet process the first swap. Treat `conn=` as lifecycle bookkeeping only; send UI updates after an explicit client message.
- The browser helper sends `{event: '__init__'}` 250ms after `htmx:wsOpen` to confirm the connection is settled before the handler sees real messages. Treat it as a best-effort readiness signal and ignore it in normal handlers.

  For relative paths, `UserWS.create()` appends `?wid=...` to the websocket URL. Independently, the browser helper includes `wid` in every JSON message, including the delayed `{event: '__init__'}` marker. Handlers should `return` early on `event == '__init__'`.
- Python → browser websocket responses should be swap-ready HTML. Browser → Python messages are JSON objects sent through `send_client`. Use stable element ids for replacement, or `hx_swap_oob` when appending/inserting into an existing target.
- Reusing a `wid` restarts its browser initiator but does not replace its already-registered FastHTML route, handler, or lifecycle hooks. Use a new `wid` and route path when those need to change.
- Separate `UserWS` instances have independent Python-side registries, but they share the FastHTML app and browser DOM. Their route paths and wid values must not overlap.

**Example**:

```python
await setup_user_ws()
```

```python
async def ws_demo(data:dict, send):
    if data.get('event') == '__init__': return
    await send(Div('Hello ' + data['msg'], id='demo-out'))

async def _conn(scope,ws,send): print('connected')
async def _disconn(scope,ws): print('disconnected')

uws = UserWS(app, srv)
wid = await uws.start('/demo-ws', ws_demo, conn=_conn, disconn=_disconn)
await uws.send_client(wid, dict(msg='world'))

await uws.stop(wid)
```

In [ ]:
allow(setup_user_ws, {UserWS: ...})

# pyskill

In [ ]:
import pote.user_ws
from pote.user_ws import setup_user_ws, UserWS

In [ ]:
list_pyskills()['pote.user_ws']

'Solveit compatible user-defined HTMX websockets.'

In [ ]:
doc(pote.user_ws)

# module pote.user_ws:

"""Solveit compatible user-defined HTMX websockets.

The browser helper in `_user_ws_js` is injected once with `setup_user_ws()`, while `UserWS(app, srv)` instances can be created many times to manage independent route/send registries.

The JS helper tracks websocket initiator elements, stores each HTMX socket wrapper on its DOM element as `el._ws`, and filters HTMX websocket events by `data-uws` attribute. Matching `htmx:wsOpen`, `htmx:wsClose`, and `htmx:wsError` events are stopped in the capture phase so Solveit’s own global websocket monitor does not mistake user websocket lifecycle events for Solveit backend failures.

`UserWS` provides Solveit-safe user HTMX websockets. One `UserWS` instance can manage many independent websocket initiators, keyed by `wid`.

Typical flow:

1. `await setup_user_ws()` once per browser session
2. `uws = UserWS(app, srv)`
3. define a FastHTML websocket handler
4. `wid = await uws.start('/ws', handler)`
5. browser → Python: `awa

In [ ]:
doc(setup_user_ws)

async def setup_user_ws(
    indicator:bool=False
):"""Inject the browser-side user_ws helper; use only in Solveit environment"""

In [ ]:
doc(UserWS)

class UserWS:
    """Manage Solveit-safe HTMX websocket initiators and server send handles keyed by `wid`"""
    def __init__(self, app, srv, create_ui=<function noop_a at 0x7b7ea8463740>, cleanup_ui=<function noop_a at 0x7b7ea8463740>): ...
    def add(self, path, handler, wid='', conn=None, disconn=None, **kwargs) -> str: ...  # Register websocket route once; optionally setup conn/disconn hooks
    def cleanup(self, wid): ...  # Remove browser initiator element
    def create(self, wid, path, **kwargs): ...  # Create browser websocket initiator for `path`
    def send(self, wid, o): ...  # Send Python to browser via stored send callable
    def send_client(self, wid, data): ...  # Send browser to Python via JS helper
    def start(self, path, handler, wid='', initkw=None, *, conn=None, disconn=None, **kwargs) -> str: ...  # Register route and create browser initiator
    def stop(self, wid): ...  # Remove browser initiator and forget server send callable
    def url(self, path, qs=''

# export -

In [ ]:
#|hide
#|eval: false
from pote.flakes import show_flakes
await show_flakes()

No warnings to report

In [ ]:
#|hide
#|eval: false
from pote.dialog import dlg_export
await dlg_export()

True